# 05-3 Gemma 모델로 텍스트 생성하기

<table align="left"><tr><td>
<a href="https://colab.research.google.com/github/rickiepark/hm-dl/blob/main/05-3-gemma-ko.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="코랩에서 실행하기"/></a>
</td></tr></table>

코랩에서 이 노트북을 실행하려면 A100 또는 High-RAM CPU 런타임을 사용해야 합니다.

## Gemma 모델로 텍스트 생성하기

In [1]:
from transformers import pipeline, set_seed

In [2]:
gemma_pipe = pipeline("text-generation", model="beomi/gemma-ko-2b")
set_seed(42)
gemma_pipe('봄이 오면', max_length=20, truncation=True)

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/555 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '봄이 오면서 피부는 피지의 과다로 인해 좀 푸석푸석하고 붉게 보이는거 같네요ㅠㅠ.. 겨울동안 엄마를 졸졸 쫓아다니며 뭐든 해주려고 애썼는데.. 그래도 피부는 건강하게 유지되어야 해서 봄맞이 팩으로 피부가 좀 좋아지길 바래봅니다! 팩하면 다들 마스크팩을 많이 하시는데요~ 저는 좀 멀리서 보면 마스크팩이 뭔지 몰라서 피부에 뭐 바르는건지 몰랐는데 말이죠~ 마스크팩이 피부를 정돈해주고 수분을 공급해주는 역할을 한다고 하네요! 그래서 팩을 사용하면 피부에 좀 생기가 돌고 건조한 피부가 촉촉해진다고 합니다! 그리고 마스크팩은 피부를 보호해주는 역할을 해서 세안을 할 때 팩으로 피부에 보호막을 만들어주는데 피부를 촉촉하게'}]

## Gemma-2 모델로 텍스트 생성하기

In [3]:
import keras
from keras import layers
import keras_nlp

2026-04-08 08:25:27.355683: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-08 08:25:29.793316: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [4]:
def make_causal_mask(seq_len):
    n_hori = keras.ops.arange(seq_len)
    n_vert = keras.ops.expand_dims(n_hori, axis=-1)
    mask = n_vert >= n_hori
    return mask

In [5]:
def make_attention_mask(padding_mask):
    # padding_mask 크기가 (2, 5)라고 가정해 보죠.
    batch_size, seq_len = keras.ops.shape(padding_mask)
    # causal_mask 크기는 (5, 5)가 됩니다.
    causal_mask = make_causal_mask(seq_len)
    # 배치 차원을 추가해 (2, 5, 5)로 만듭니다.
    causal_mask = keras.ops.broadcast_to(causal_mask, (batch_size, seq_len, seq_len))
    # 브로드캐스팅을 위해 padding_mask 크기를 (2, 1, 5)로 만듭니다.
    padding_mask = keras.ops.expand_dims(padding_mask, axis=1)
    return keras.ops.minimum(causal_mask, padding_mask)

In [6]:
class AttentionMask(keras.Layer):
    def call(self, padding_mask):
        return make_attention_mask(padding_mask)

In [ ]:
from keras_nlp.src.models.gemma.gemma_attention import CachedGemmaAttention
from keras_nlp.src.models.gemma.rms_normalization import RMSNormalization

def gemma2_decoder(x, padding_mask, num_query_heads, num_key_value_heads,
                  interm_dim, hidden_dim, head_dim):
    # 어텐션 마스크를 계산합니다.
    attention_mask = AttentionMask()(padding_mask)
    # 스킵 연결을 준비합니다.
    residual = x
    x = RMSNormalization()(x)
    # 멀티 헤드 어텐션을 통과합니다.
    gemma_attention = CachedGemmaAttention(head_dim=head_dim,
                                           num_query_heads=num_query_heads,
                                           num_key_value_heads=num_key_value_heads,
                                           use_sliding_window_attention=True,
                                           dropout=0.0)
    x = gemma_attention(x, attention_mask)
    # 포스트 정규화
    x = RMSNormalization()(x)
    # 스킵 연결
    x = x + residual
    # 스킵 연결을 준비합니다.
    residual = x
    # 위치별 피드 포워드 네트워크
    x = RMSNormalization()(x)
    x1 = layers.Dense(interm_dim, activation='gelu', use_bias=False)(x)
    x2 = layers.Dense(interm_dim, use_bias=False)(x)
    x = x1 * x2
    x = layers.Dense(hidden_dim, use_bias=False)(x)
    # 포스트 정규화
    x = RMSNormalization()(x)
    # 스킵 연결
    x = x + residual
    return x

: 

In [ ]:
from keras_nlp.layers import ReversibleEmbedding

# Gemma2 2B
vocab_size = 256000
num_layers = 26
num_query_heads = 8
num_key_value_heads = 4
interm_dim = 9216
hidden_dim = 2304
head_dim = 256

token_ids = keras.Input(shape=(None,))
padding_mask = keras.Input(shape=(None,))

token_embedding_layer = ReversibleEmbedding(vocab_size, hidden_dim)
x = token_embedding_layer(token_ids)
x = layers.Lambda(lambda x: x * keras.ops.sqrt(hidden_dim))(x)

for _ in range(num_layers):
    x = gemma2_decoder(x, padding_mask, num_query_heads, num_key_value_heads,
                      interm_dim, hidden_dim, head_dim)

x = RMSNormalization()(x)
outputs = token_embedding_layer(x, reverse=True)
model = keras.Model(inputs=(token_ids, padding_mask),
                    outputs=(outputs))
model.summary(line_length=100)

2026-04-08 08:25:31.175044: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2026-04-08 08:25:31.213792: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 2359296000 exceeds 10% of free system memory.
2026-04-08 08:25:33.464878: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 2359296000 exceeds 10% of free system memory.
2026-04-08 08:25:34.694771: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 2359296000 exceeds 10% of free system memory.
2026-04-08 08:25:38.248322: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 84934656 exceeds 10% of free system memory.
2026-04-08 08:25:38.289731: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 84934656 exceeds 10% of free system memory.


In [ ]:
!mkdir ~/.kaggle/
!mv kaggle.json ~/.kaggle/

In [ ]:
gemma = keras_nlp.models.GemmaCausalLM.from_preset('gemma2_2b_en')

In [ ]:
sampler = keras_nlp.samplers.TopPSampler(p=0.8, seed=42)
gemma.compile(sampler=sampler)
gemma.generate('봄이 오면', max_length=20)

'봄이 오면 많은 사람들이 물론 대부분 자외선 차단제를'